# BlueField DPUs
BlueField is a next-generation Data Processing Unit (DPU) developed by NVIDIA (formerly Mellanox Technologies). It integrates powerful computing, networking, and storage acceleration capabilities into a single device. Designed for modern data centers, BlueField DPUs offer hardware-accelerated data processing, efficient resource management, and enhanced security features.

## Key Features of BlueFields
- Integrated Compute and Networking:
  - Combines an ARM-based SoC (System-on-Chip) with a ConnectX-6 Dx network interface controller (NIC).
  - Offers hardware acceleration for networking and storage operations.

- Advanced Networking:
  - Supports up to 200Gb/s Ethernet or InfiniBand networking.
  - Equipped with SR-IOV, RDMA, and GPUDirect Storage capabilities.
    
- Storage Acceleration:
  - Enables offloading for NVMe over Fabrics (NVMe-oF).
  - Provides support for RAID and data mirroring.

- Security Capabilities:
  - Includes a hardware root of trust and secure boot features.
  -  Offers real-time encryption, data isolation, and zero-trust security models.

## Create a slice using BlueField SmartNICs 

This notebook shows how to create an isolated local Ethernet using BlueField Smart NICs and connect compute nodes to it and use FABlib's automatic configuration functionality.


## Import the FABlib Library


In [1]:
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
                     
fablib.show_config();

User: choueiri@email.sc.edu bastion key is valid!
Configuration is valid


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Token File,/home/fabric/.tokens.json
Project ID,8eaa3ec2-65e7-49a3-8c09-e1761141a6ad
Bastion Host,bastion.fabric-testbed.net
Bastion Username,choueiri_0000118746
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub
Slice Private Key File,/home/fabric/work/fabric_config/slice_key


## Create the Experiment Slice

This example sets up two nodes, each equipped with a BlueField NIC, connected to an isolated local Ethernet. 

Each node is created with a single NIC component, utilizing the `NIC_ConnectX_7_400` and `NIC_Basic` models. These components are attached to the node via PCI passthrough. A list of other available NIC models is provided below. To retrieve the interfaces associated with a NIC component, use the `get_interfaces()` method. Many dedicated NICs feature multiple ports, either of which can be connected to the network.

The node connected to the BlueField SmartNIC runs the `dpu_ubuntu_24` image, which includes DOCA version `2.9.1` by default. Alternatively, users can deploy VMs with the `default_ubuntu_24` image and manually install a different DOCA version as needed.

For automatic configuration, specify a subnet for the network and set the interface mode to `auto` using `iface1.set_mode('auto')` before submitting the request. With this setup, FABlib assigns an IP address from the subnet and configures the device during post-boot setup. Additionally, routes can be pre-configured before submitting the request.

### Available NIC Component Models:
- **NIC_Basic**: 100 Gbps Mellanox ConnectX-6 SR-IOV VF (1 Port)
- **NIC_ConnectX_5**: 25 Gbps Dedicated Mellanox ConnectX-5 PCI Device (2 Ports)
- **NIC_ConnectX_6**: 100 Gbps Dedicated Mellanox ConnectX-6 PCI Device (2 Ports)
- **NIC_ConnectX_7_100**: 100 Gbps Dedicated Mellanox BlueField-3 ConnectX-7 PCI Device (2 Ports)
- **NIC_ConnectX_7_400**: 400 Gbps Dedicated Mellanox BlueField-3 ConnectX-7 PCI Device (2 Ports)
- **NIC_BlueField2_ConnectX_6**: 100 Gbps Dedicated Mellanox BlueField-2 ConnectX-6 PCI Device (2 Ports)

In [2]:
slice_name = 'BF3'
#site = fablib.get_random_site()
site="NCSA"
print(f"Site: {site}")

node1_name = 'Node1'
node2_name = 'Node2'
node3_name = 'Node3'

network1_name='net1'
network2_name='net2'

node1_image = "default_ubuntu_20"
node2_image = "dpu_ubuntu_24"
node3_image = "default_ubuntu_20"

Site: NCSA


In [3]:
#Create Slice
slice = fablib.new_slice(name=slice_name)

# Network
net1 = slice.add_l3network(name=network1_name)
net2 = slice.add_l3network(name=network2_name)

# Node1
node1 = slice.add_node(name=node1_name, site=site, cores=16, ram=32, disk=200, image=node1_image)
iface1_1 = node1.add_component(model='NIC_Basic', name='nic1').get_interfaces()[0]
iface1_1.set_mode('auto')
net1.add_interface(iface1_1)

# Node2
node2 = slice.add_node(name=node2_name, site=site, cores=16, ram=32, disk=200, image=node2_image)
iface2_1 = node2.add_component(model='NIC_ConnectX_7_100', name='nic2').get_interfaces()[0]
iface2_1.set_mode('auto')
net1.add_interface(iface2_1)

iface2_2 =  node2.get_component(name='nic2').get_interfaces()[1]
iface2_2.set_mode('auto')
net2.add_interface(iface2_2)

# Node3
node3 = slice.add_node(name=node3_name, site=site, cores=16, ram=32, disk=200, image=node3_image)
iface3_1 = node3.add_component(model='NIC_Basic', name='nic3').get_interfaces()[0]
iface3_1.set_mode('auto')
net2.add_interface(iface3_1)

#Submit Slice Request
slice.submit();


Retry: 11, Time: 266 sec


ID,fbdff169-08d3-4ff8-832f-4f80803b6188
Name,BF3
Lease Expiration (UTC),2026-05-27 16:38:07 +0000
Lease Start (UTC),2026-05-26 16:38:07 +0000
Project ID,8eaa3ec2-65e7-49a3-8c09-e1761141a6ad
State,StableOK
Email,CHOUEIRI@email.sc.edu
UserId,78504735-b34f-42a8-be20-8e71d6acf13e


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
58cc61cf-744b-4447-a487-0ea80522adc0,Node1,16,32,500,default_ubuntu_20,qcow2,ncsa-w3.fabric-testbed.net,NCSA,ubuntu,2620:0:c80:1001:f816:3eff:fe5f:2de3,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2620:0:c80:1001:f816:3eff:fe5f:2de3,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
d23210b8-771a-4237-86d3-19126d742970,Node2,16,32,500,dpu_ubuntu_24,qcow2,ncsa-w2.fabric-testbed.net,NCSA,ubuntu,2620:0:c80:1001:f816:3eff:feb7:55af,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2620:0:c80:1001:f816:3eff:feb7:55af,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
78b89e86-3f51-4f2e-8b33-4e934142a640,Node3,16,32,500,default_ubuntu_20,qcow2,ncsa-w3.fabric-testbed.net,NCSA,ubuntu,2620:0:c80:1001:f816:3eff:feda:a41f,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2620:0:c80:1001:f816:3eff:feda:a41f,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key


ID,Name,Layer,Type,Site,Subnet,Gateway,State,Error
817c05e1-d7e9-4b4f-8259-5b8bd77e9bd9,net1,L3,FABNetv4,NCSA,10.132.129.0/24,10.132.129.1,Active,
00c6a8d2-68d1-44c8-a5fb-8d382dac4736,net2,L3,FABNetv4,NCSA,10.132.130.0/24,10.132.130.1,Active,


Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
Node1-nic1-p1,p1,Node1,net1,100,auto,,06:71:DB:A8:AD:43,enp7s0,enp7s0,10.132.129.3,4,HundredGigE0/0/0/13
Node2-nic2-p2,p2,Node2,net2,100,auto,,E8:9E:49:4E:FE:81,enp8s0np1,enp8s0np1,10.132.130.3,7,HundredGigE0/0/0/3
Node2-nic2-p1,p1,Node2,net1,100,auto,,E8:9E:49:4E:FE:80,enp7s0np0,enp7s0np0,10.132.129.2,7,HundredGigE0/0/0/1
Node3-nic3-p1,p1,Node3,net2,100,auto,,06:DD:4B:2F:DE:C1,enp7s0,enp7s0,10.132.130.2,4,HundredGigE0/0/0/13



Time to print interfaces 266 seconds


## Configure the Bluefield Smart NIC

The BlueField SmartNIC is configured by assigning the private IP address `192.168.100.1` to the `tmfifo_net0` device, enabling communication and management of the BlueField DPU. Additionally, the BlueField bundle (BFB) is installed on the DPU via the designated RShim interface, ensuring firmware updates and configuration optimizations for enhanced data center performance.

When using the `dpu_ubuntu_24` image for a node (e.g., `Node1`) connected to the BlueField SmartNIC, the BFB image is available by default at:  
`/opt/bf-bundle/bf-bundle-2.9.1-40_24.11_ubuntu-22.04_prod.bfb`.  
The installation process is initiated when `bluefield.configure()` is executed.

To run custom commands, provide a list of command strings as an argument to `bluefield.configure(commands)`.

### Accessing DPU:
#### SSH Access:
To SSH into DPU from the VM, use:
```
ssh ubuntu@192.168.100.2
```
After pushing the BFB image to DPU, the default credentials are:  
- **Username:** `ubuntu`  
- **Password:** `ubuntu` (you will be prompted to change it upon first login)

#### Console Access:
If SSH access is unavailable, you can connect to DPU via the console interface:
```
screen /dev/rshim0/console
```
This method provides an alternative way to manage the SmartNIC if SSH connectivity is lost.


In [13]:
slice = fablib.get_slice(slice_name)

node2 = slice.get_node(name=node2_name) 
bluefield = node2.get_component(name='nic2')
output = bluefield.configure()

Checking if local host has root access...
Checking if rshim driver is running locally...
Warn: 'pv' command not found. Continue without showing BFB progress.
Pushing bfb
 INFO[PSC]: PSC BL1 START
 INFO[BL2]: start
 INFO[BL2]: boot mode (rshim)
 INFO[BL2]: VDD_CPU: 750 mV
 INFO[BL2]: VDDQ: 1118 mV
 INFO[BL2]: DDR POST passed
 INFO[BL2]: UEFI loaded
 INFO[BL31]: start
 INFO[BL31]: lifecycle GA Secured
 INFO[BL31]: RPMB Key NOT programmed
 INFO[BL31]: runtime
 INFO[BL31]: MB ping success
 INFO[UEFI]: eMMC init
 INFO[UEFI]: eMMC probed
 INFO[UEFI]: UPVS valid
 INFO[UEFI]: PMI: updates started
 INFO[UEFI]: PMI: total updates: 1
 INFO[UEFI]: PMI: updates completed, status 0
 INFO[UEFI]: PCIe enum start
 INFO[UEFI]: PCIe enum end
 INFO[UEFI]: UEFI Secure Boot (enabled)
 INFO[UEFI]: Redfish enabled
 INFO[UEFI]: exit Boot Service
 INFO[MISC]: Erasing eMMC drive: /dev/mmcblk0
 INFO[MISC]: Erasing NVME drive: /dev/nvme0n1
 INFO[MISC]: Ubuntu installation started
 INFO[MISC]: Installing OS image
 

### Re-apply the network config post re-imaging

Reapply the network configuration on the Node connected to the BlueField. This is only effective when using automatic configuration, where interfaces are set up via `iface1.set_mode('auto')` or `iface1.set_mode('config')`.

In [14]:
node2.config()
slice.list_interfaces();

Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
Node1-nic1-p1,p1,Node1,net1,100,auto,,02:08:A4:D8:FF:43,enp7s0,enp7s0,10.137.129.3,4,HundredGigE0/0/0/5
Node2-nic2-p1,p1,Node2,net1,100,auto,,CC:40:F3:8F:03:0A,enp7s0np0,enp7s0np0,10.137.129.2,7,HundredGigE0/0/0/16
Node2-nic2-p2,p2,Node2,net2,100,auto,,CC:40:F3:8F:03:0B,enp8s0np1,enp8s0np1,10.137.131.3,7,HundredGigE0/0/0/17
Node3-nic3-p1,p1,Node3,net2,100,auto,,0E:5E:49:B6:89:AF,enp7s0,enp7s0,10.137.131.2,4,HundredGigE0/0/0/7


## Run the Experiment

With automatic configuration the slice is ready for experimentation after it becomes active.  Note that automatic configuration works well when saving slices to a file and reinstantiating the slice.  Configuration tasks can be stored in the saved slice, reducing the complexity of notebooks and other runtime steps.

We will find the ping round trip time for this pair of sites.  Your experiment should be more interesting!

**Note:** If the ping fails, try rerunning the cell above to ensure the IP address is properly configured, then attempt the ping again.

In [1]:
from ipaddress import ip_address, IPv4Address, IPv6Address, IPv4Network, IPv6Network
import ipaddress

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()

slice_name = 'ETC_BF3'

slice = fablib.get_slice(slice_name)

node1_name = 'Node1'
node2_name = 'Node2'
node3_name = 'Node3'

node1 = slice.get_node(name=node1_name)        
node2 = slice.get_node(name=node2_name)           
node3 = slice.get_node(name=node3_name)           

print(node1)
print(node2)
print(node3)

User: Amithgspn@sc.edu bastion key is valid!
Configuration is valid
-----------------  ------------------------------------------------------------------------------------------------------------------------------------------
ID                 a3eb067e-6c61-4030-9a89-9e7728695dcc
Name               Node1
Cores              16
RAM                32
Disk               500
Image              default_ubuntu_24
Image Type         qcow2
Host               mass-w1.fabric-testbed.net
Site               MASS
Management IP      2001:48e8:6401:3:f816:3eff:fe74:36b3
Reservation State  Active
Error Message
SSH Command        ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:48e8:6401:3:f816:3eff:fe74:36b3
-----------------  ------------------------------------------------------------------------------------------------------------------------------------------
-----------------  ---------------------------------------------------------------

## Appending list of nodes
All nodes are appended to a list to execute commands in parallel

In [5]:
nodes = []

nodes.append(slice.get_node(name="Node1"))     
nodes.append(slice.get_node(name="Node2"))
nodes.append(slice.get_node(name="Node3"))

node1 = nodes[0]
node2 = nodes[1]
node3 = nodes[2]

## NAT64 setup
The code below checks if an IPv6 address is available to set up NAT64. We will upload the script [scripts/nat64.sh](./scripts/nat64.sh) to the all nodes and execute it

In [8]:
from ipaddress import ip_address, IPv6Address

threads = []

for node in nodes:
    if type(ip_address(node.get_management_ip())) is IPv6Address:
        node.upload_file('scripts/nat64.sh', 'nat64.sh')
        threads.append(node.execute_thread(f'chmod +x nat64.sh && ./nat64.sh'))

for thread in threads:
    thread.result()

## Installing dependencies
The code below installs packages that are prerequisites to the upcoming installations and needed to run the experiments

In [9]:
threads = []

for node in nodes:
    threads.append(node.execute_thread('''
        sudo apt-get update;
            sudo apt-get install -y build-essential python3-pip python3-pyelftools libnuma-dev pkg-config net-tools hping3;
        sudo pip3 install meson ninja
    '''))

for thread in threads:
    thread.result()

In [11]:
threads = []

for node in nodes:
    threads.append(node.execute_thread('''
        sudo apt install python3.12-venv
        python3 -m venv .venv
        
        # activate venv
        source .venv/bin/activate
        
        # upgrade pip & install packages
        python3 -m pip install --upgrade pip
        pip install numpy
        pip install pandas
        pip install scikit-learn
        pip install netaddr
        
        # verify numpy version
        python -c "import numpy as np; print('numpy', np.__version__)"
    '''))

for thread in threads:
    thread.result()


## DPDK version
DPDK version: MLNX_DPDK 22.11.2410.1.0

## Installing Custom DPDK
The code below downloads, builds, and installs DPDK on all servers

In [5]:
threads = []

for node in nodes:
    threads.append(node.execute_thread('''
        git clone https://github.com/DPDK/dpdk.git;
        cd dpdk;
        sudo meson build;
        cd build;
        sudo ninja;
        sudo ninja install; 
        sudo ldconfig
    '''))

for thread in threads:
    thread.result()

In [6]:
stdout, stderr = node2.execute(f'cd dpdk/examples/pipeline && sudo make', quiet=True)

## Installing Pktgen
In this project, we will send from node 1 packets at a high rate to node 2 using a DPDK-based packet generation tool called pktgen [5]. The code below downloads and installs Pktgen-DPDK on all nodes

In [7]:
threads = []

for node in nodes:
    threads.append(node.execute_thread('''
        sudo git clone https://github.com/pktgen/Pktgen-DPDK; 
        sudo sed -i \"s/deps += \\[dependency('numa', required: true)\\]/deps += \\[dependency('numa', required: false)\\]/\" /home/ubuntu/Pktgen-DPDK/app/meson.build;
        sudo apt-get install -y cmake libpcap-dev libbsd-dev;
        cd Pktgen-DPDK &&  sudo meson build && sudo ninja -C build && cd build/ && sudo meson install
    '''))

for thread in threads:
    thread.result()

## Build pipeline library
The code below builds the DPDK pipeline library in node 2 (on which the pipeline will be running) to put all its functions into effect 

In [8]:
stdout, stderr = node2.execute(f'cd dpdk/examples/pipeline && sudo make', quiet=True)

## Install p4c
The code below downloads and installs the p4c compiler needed to compile the p4 code into a DPDK pipeline. In this project, p4c is built from a version where the architecture has been modified.

In [35]:
stdout, stderr = node2.execute('[ -d p4c ] || git clone https://github.com/CILab-USC/p4c.git', quiet = True)
stdout, stderr = node2.execute('sudo DEBIAN_FRONTEND=noninteractive apt-get update && sudo DEBIAN_FRONTEND=noninteractive apt-get install -y cmake g++ git automake libtool libgc-dev bison flex libfl-dev libboost-dev libboost-iostreams-dev libboost-graph-dev llvm pkg-config python3 python3-pip python3.12-venv tcpdump || echo "apt-get failed or is disallowed; continuing without installing python3-venv (you may need it)."', quiet = True)
stdout, stderr = node2.execute('cd p4c && git fetch origin && git pull --ff-only || git reset --hard origin/$(git rev-parse --abbrev-ref HEAD) || true && (python3 -m venv .venv || (echo "python3 -m venv failed; trying ensurepip fallback"; python3 -m ensurepip --upgrade && python3 -m venv .venv)) && . .venv/bin/activate && python -m pip install --upgrade pip setuptools wheel && pip install -r requirements.txt && mkdir -p build && cd build && cmake .. && make -j4 && sudo make install || echo "sudo make install failed or was not permitted; build artifacts remain in build/."', quiet = True)

Hit:1 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:2 http://nova.clouds.archive.ubuntu.com/ubuntu noble InRelease
Hit:3 http://nova.clouds.archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:4 http://nova.clouds.archive.ubuntu.com/ubuntu noble-backports InRelease
Ign:5 https://linux.mellanox.com/public/repo/doca/2.9.1/ubuntu24.04/x86_64 ./ InRelease
Hit:6 https://linux.mellanox.com/public/repo/doca/2.9.1/ubuntu24.04/x86_64 ./ Release
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
cmake is already the newest version (3.28.3-1build7).
g++ is already the newest version (4:13.2.0-7ubuntu1).
git is already the newest version (1:2.43.0-1ubuntu7.3).
automake is already the newest version (1:1.16.5-1.3ubuntu1).
libtool is already the newest version (2.4.7-7build1).
libgc-dev is already the newest version (1:8.2.6-1build1).
bison is already the newest version (2:3.8.2+dfsg-1build2).
flex is already the newest versio

## Reboot
After installing the ConnectX Mellanox divers, it is essential to reboot the nodes to complete the installation process or to ensure that updates are applied correctly

In [ ]:
for node in nodes:
    node.os_reboot()

## Get interfaces names
In this step we will get the interface names so that we can assign IP addresses to them. 

In [36]:
node1_iface = node1.get_interface(network_name='net1') 
node1_iface_name = node1_iface.get_device_name() 
print(f'node1_iface: {node1_iface_name}')

node2_iface1 = node2.get_interface(network_name='net1') 
node2_iface1_name = node2_iface1.get_device_name() + 'np0'
node2_iface1_name = 'enp7s0np0'
print(f'node2_iface1: {node2_iface1_name}')

node2_iface2 = node2.get_interface(network_name='net2') 
node2_iface2_name = node2_iface2.get_device_name() + 'np1'
node2_iface2_name = 'enp8s0np1'
print(f'node2_iface2: {node2_iface2_name}')

node3_iface = node3.get_interface(network_name='net2') 
node3_iface_name = node3_iface.get_device_name()
print(f'node3_iface: {node3_iface_name}')

node1_iface: enp7s0
node2_iface1: enp7s0np0
node2_iface2: enp8s0np1
node3_iface: enp7s0


## Turning all interfaces up
In this step, we will use the ip link command to turn the interfaces up

In [37]:
print(node1_iface_name)
print(node2_iface1_name)
print(node2_iface2_name)
print(node3_iface_name)

stdout, stderr = node1.execute(f'sudo ip link set dev {node1_iface_name} up', quiet=True)
stdout, stderr = node2.execute(f'sudo ip link set dev {node2_iface1_name} up', quiet=True)
stdout, stderr = node2.execute(f'sudo ip link set dev {node2_iface2_name} up', quiet=True)
stdout, stderr = node3.execute(f'sudo ip link set dev {node3_iface_name} up', quiet=True)

enp7s0
enp7s0np0
enp8s0np1
enp7s0


## SHardcode MAC addresses
For simplicity, we will use the following MAC addresses for the interfaces:
<ul>
    <li> node1_iface_MAC = '00:00:00:00:00:01' (shown as 00:01 in the figure below) </li>
    <li>node2_iface1_MAC = '00:00:00:00:00:21' (shown as 00:21 in the figure below)</li>
    <li>node2_iface2_MAC = '00:00:00:00:00:22' (shown as 00:22 in the figure below)</li>
    <li>node3_iface_MAC = '00:00:00:00:00:03' (shown as 00:03 in the figure below)</li>
</ul>

In [38]:
node1_iface_MAC = '00:00:00:00:00:01'
node2_iface1_MAC = '00:00:00:00:00:21'
node2_iface2_MAC = '00:00:00:00:00:22'
node3_iface_MAC = '00:00:00:00:00:03'

## Configuring the IP and MAC addresses on node1_iface and node2_iface1

We will use the network 192.168.10.0/24 between node1 and node2. We will assign the IP address 192.168.10.1 to node1's interface and 192.168.10.2 to its neighboring interface on node2.

In [39]:
node1_node2_subnet = "192.168.10.0/24"
node1_ip = '192.168.10.1/24'
node2_1_ip = '192.168.10.2/24'

stdout, stderr = node1.execute(f'sudo ifconfig {node1_iface_name} {node1_ip}')
stdout, stderr = node2.execute(f'sudo ifconfig {node2_iface1_name} {node2_1_ip}')

stdout, stderr = node1.execute(f'sudo ifconfig {node1_iface_name} hw ether {node1_iface_MAC}')
stdout, stderr = node2.execute(f'sudo ifconfig {node2_iface1_name} hw ether {node2_iface1_MAC}')

In [40]:
stdout, stderr = node1.execute(f'ifconfig -a')
stdout, stderr = node2.execute(f'ifconfig -a')
stdout, stderr = node3.execute(f'ifconfig -a')

enp3s0: flags=4163<UP,BROADCAST,RUNNING,MULTICAST>  mtu 9000
        inet 10.30.6.97  netmask 255.255.254.0  broadcast 10.30.7.255
        inet6 2001:48e8:6401:3:f816:3eff:fe74:36b3  prefixlen 64  scopeid 0x0<global>
        inet6 fe80::f816:3eff:fe74:36b3  prefixlen 64  scopeid 0x20<link>
        ether fa:16:3e:74:36:b3  txqueuelen 1000  (Ethernet)
        RX packets 4036  bytes 524889 (524.8 KB)
        RX errors 0  dropped 0  overruns 0  frame 0
        TX packets 1381  bytes 175103 (175.1 KB)
        TX errors 0  dropped 0 overruns 0  carrier 0  collisions 0

enp7s0: flags=4163<UP,BROADCAST,RUNNING,MULTICAST>  mtu 1500
        inet 192.168.10.1  netmask 255.255.255.0  broadcast 192.168.10.255
        inet6 fe80::8af:e7ff:fece:d2c0  prefixlen 64  scopeid 0x20<link>
        ether 00:00:00:00:00:01  txqueuelen 1000  (Ethernet)
        RX packets 4  bytes 392 (392.0 B)
        RX errors 0  dropped 0  overruns 0  frame 0
        TX packets 21  bytes 1678 (1.6 KB)
        TX errors 0  dr

## Configuring the IP and MAC addresses on node2_iface2 and node3_iface

We will use the network 192.168.20.0/24 between node2 and node3. We will assign the IP address 192.168.20.1 to node3's interface and 192.168.20.2 to its neighboring interface on node2.

In [41]:
node2_node3_subnet = "192.168.20.0/24"
node2_2_ip = '192.168.20.2/24'
node3_ip = '192.168.20.1/24'

stdout, stderr = node2.execute(f'sudo ifconfig {node2_iface2_name} {node2_2_ip}')
stdout, stderr = node3.execute(f'sudo ifconfig {node3_iface_name} {node3_ip}')

stdout, stderr = node2.execute(f'sudo ifconfig {node2_iface2_name} hw ether {node2_iface2_MAC}')
stdout, stderr = node3.execute(f'sudo ifconfig {node3_iface_name} hw ether {node3_iface_MAC}')

## Enable forwarding on node2

The command "sudo sysctl -w net.ipv4.ip_forward=1" is used to enable IP forwarding on a Linux system.

IP forwarding is a feature that allows a system to act as a router by forwarding network packets from one network interface to another. By default, IP forwarding is usually disabled on Linux systems for security reasons. 

The command will be executed on the node2.

In [42]:
stdout, stderr = node2.execute(f'sudo sysctl -w net.ipv4.ip_forward=1', quiet=True)
stdout, stderr = node3.execute(f'sudo sysctl -w net.ipv4.ip_forward=1', quiet=True)


## Configure ARP and static routing

In this step, we will configure static ARP entries and static routing on node1, node2, and node3.

In [43]:
ip1 = node1_ip.split('/')[0]
ip2_1 = node2_1_ip.split('/')[0]
ip2_2 = node2_2_ip.split('/')[0]
ip3 = node3_ip.split('/')[0]
subnet1 = "192.168.10.0/24"
subnet2 = "192.168.20.0/24"

stdout, stderr = node1.execute(f'sudo arp -s {ip2_1} {node2_iface1_MAC}')
stdout, stderr = node1.execute(f'sudo ip route add {subnet2} via {ip2_1}')

stdout, stderr = node2.execute(f'sudo arp -s {ip1} {node1_iface_MAC}')
stdout, stderr = node2.execute(f'sudo arp -s {ip3} {node3_iface_MAC}')

stdout, stderr = node3.execute(f'sudo arp -s {ip2_2} {node2_iface2_MAC}')
stdout, stderr = node3.execute(f'sudo ip route add {subnet1} via {ip2_2}')

RTNETLINK answers: File exists
RTNETLINK answers: File exists


## OVS e-switch enable

We will need to enable the wire ports from Bluefield-3 to the host. Whatever is received on ports 0 and 1 of Bluefield-3 is sent to the host of the Node 2 server.

In [15]:
# Delete the OVS bridges (residue)
ovs-vsctl del-br ovsbr1
ovs-vsctl del-br ovsbr2

# Add the OVS bridges
ovs-vsctl add-br ovsbr1
ovs-vsctl add-br ovsbr2

# Assign p0 and pf0hpf ports to ovsbr1
ovs-vsctl add-port ovsbr1 p0;
ovs-vsctl add-port ovsbr1 pf0hpf;

ovs-vsctl add-port ovsbr2 p1;
ovs-vsctl add-port ovsbr2 pf1hpf;

# Add flood flow
sudo ovs-ofctl add-flow ovsbr1 arp,actions=FLOOD
sudo ovs-ofctl add-flow ovsbr2 arp,actions=FLOOD
# If we want to have p0 as ovsbr1
sudo ovs-ofctl add-flow ovsbr1 in_port=p0,actions=output:pf0hpf
sudo ovs-ofctl add-flow ovsbr1 in_port=pf0hpf,actions=output:p0

sudo ovs-ofctl add-flow ovsbr2 in_port=p1,actions=output:pf1hpf
sudo ovs-ofctl add-flow ovsbr2 in_port=pf1hpf,actions=output:p1

# Set Open_v_Swtich
ovs-vsctl set Open_vSwitch . other_config:hw-offload=true;

# Restart and enable openvswitch
systemctl restart openvswitch-switch
systemctl enable openvswitch-switch

SyntaxError: invalid syntax (1856706164.py, line 2)

## Mellanox devices

In this step, we will inspect and start all Mellanox devices.

In [47]:
stdout, stderr = node1.execute(f'sudo ibdev2netdev')
stdout, stderr = node1.execute(f'sudo mst status', quiet=True)
stdout, stderr = node1.execute(f'sudo mst start', quiet=True)
stdout, stderr = node1.execute(f'sudo mst status', quiet=True)

stdout, stderr = node2.execute(f'sudo ibdev2netdev')
stdout, stderr = node2.execute(f'sudo mst status', quiet=True)
stdout, stderr = node2.execute(f'sudo mst start', quiet=True)
stdout, stderr = node2.execute(f'sudo mst status', quiet=True)

stdout, stderr = node3.execute(f'sudo ibdev2netdev')
stdout, stderr = node3.execute(f'sudo mst status', quiet=True)
stdout, stderr = node3.execute(f'sudo mst start', quiet=True)
stdout, stderr = node3.execute(f'sudo mst status', quiet=True)

sudo: ibdev2netdev: command not found
sudo: ibdev2netdev: command not found


In [45]:
threads = []

for node in nodes:
    threads.append(node.execute_thread(f' sudo sh -c  "echo 2048 > /sys/kernel/mm/hugepages/hugepages-2048kB/nr_hugepages"'))

for thread in threads:
    thread.result()

## Ubuntu version 
The Ubuntu version is 24. The Linux kernel version is "6.8.0-88-generic."

We need to install extra Linux modules to add Macsec, then load mlx5_ib

In [ ]:
# show whether a macsec module file exists for the running kernel
stdout, stderr = node2.execute(f'find /lib/modules/$(uname -r) -type f -name 'macsec*' -print || true', quiet=True)

# try to show modinfo if installed
stdout, stderr = node2.execute(f'modinfo macsec 2>/dev/null || echo "macsec modinfo: not present"', quiet=True)

stdout, stderr = node2.execute(f'sudo apt-get update', quiet=True)
stdout, stderr = node2.execute(f'sudo apt-get install -y linux-modules-extra-$(uname -r)', quiet=True)

# load macsec
stdout, stderr = node2.execute(f'sudo modprobe macsec', quiet=True)

# now try mlx5_ib again
stdout, stderr = node2.execute(f'sudo modprobe mlx5_ib', quiet=True)

# check loaded modules and verbs devices
stdout, stderr = node2.execute(f'lsmod | egrep 'macsec|mlx5_core|mlx5_ib' || true', quiet=True)
stdout, stderr = node2.execute(f'sudo dmesg | egrep -i 'mlx5|macsec|ib|rdma' | tail -n 80', quiet=True)
stdout, stderr = node2.execute(f'ibv_devices || true', quiet=True)
stdout, stderr = node2.execute(f'sudo /home/ubuntu/dpdk/usertools/dpdk-devbind.py --status || true', quiet=True)

## Delete the Slice

Please delete your slice when you are done with your experiment.

In [4]:
slice = fablib.get_slice(slice_name)
slice.delete()